# 01 — Data: download, EDA, augmentation preview

Downloads NCT-CRC-HE-100K (train) + CRC-VAL-HE-7K (val) into `PERSIST_DIR/data` (once),
shows the class distribution, and previews the Albumentations pipeline incl. `HEDStainJitter`.

In [ ]:
%run shared.ipynb
import os
from pathlib import Path
PERSIST_DIR = Path(os.environ.get("PERSIST_DIR", "/workspace/shared/ft004"))
set_seed(42)


In [ ]:
# ~10 GB download on first run; cached thereafter. Resumes are no-ops.
paths = ensure_dataset(PERSIST_DIR)
TRAIN_DIR = paths["NCT-CRC-HE-100K"]
VAL_DIR = paths["CRC-VAL-HE-7K"]
print("train:", TRAIN_DIR)
print("val  :", VAL_DIR)


In [ ]:
# Class distribution
import matplotlib.pyplot as plt
train_ds = PathologyPatchDataset(TRAIN_DIR, transform=None)
val_ds = PathologyPatchDataset(VAL_DIR, classes=train_ds.classes, transform=None)

from collections import Counter
ctr = Counter(lbl for _, lbl in train_ds.samples)
counts = [ctr[i] for i in range(len(train_ds.classes))]
plt.figure(figsize=(10, 4))
plt.bar(train_ds.classes, counts, color="#4C78A8")
plt.title(f"NCT-CRC-HE-100K class distribution (train={len(train_ds)}, val={len(val_ds)})")
plt.ylabel("patches"); plt.tight_layout(); plt.show()
print({c: ctr[i] for i, c in enumerate(train_ds.classes)})


In [ ]:
# Augmentation preview: original vs train-aug (with stain) vs strong OOD shift
import numpy as np, matplotlib.pyplot as plt
from PIL import Image

train_aug = build_train_aug(use_stain_aug=True, theta=0.05)
ood_aug = build_ood_aug(theta=0.15)

idxs = [next(i for i, (_, l) in enumerate(train_ds.samples) if l == k)
        for k in range(len(train_ds.classes))]

fig, axes = plt.subplots(3, len(idxs), figsize=(2.0 * len(idxs), 6.5))
for col, i in enumerate(idxs):
    path, lbl = train_ds.samples[i]
    img = np.array(Image.open(path).convert("RGB"))
    aug = denormalize(train_aug(image=img)["image"])
    ood = denormalize(ood_aug(image=img)["image"])
    for row, (im, tag) in enumerate([(img, "orig"), (aug, "train-aug"), (ood, "OOD shift")]):
        axes[row, col].imshow(im); axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(train_ds.classes[lbl], fontsize=9)
        if col == 0:
            axes[row, col].set_ylabel(tag, fontsize=10)
plt.suptitle("Augmentation pipeline preview", y=1.02)
plt.tight_layout(); plt.show()
